# Pipeline de Preprocesamiento de Datasets de Caídas

**Objetivo:** Pipeline de extracción, transformación y carga (ETL) para procesar los datos inmutables en la capa `bronce`, realizar la validación de integridad y filtrado de calidad de resampleo en la capa `plata`, y generar los archivos Parquet resampleados a 50 Hz en la capa `oro`.

### Arquitectura de Almacenamiento
* **`bronce/falls`**: CSVs crudos e inmutables de los datasets (`SisFall`, `FallAllD`, `KFall`, `UPFall`, `UMAFall`).
* **`plata/falls`**: Métricas de calidad por trial y configuraciones JSON de filtrado.
* **`oro/falls`**: Datasets finales en formato Parquet a 50 Hz con el esquema estandarizado de 14 columnas.

### Estándar de Unidades Físicas y Ejes
* **Acelerómetros (`Ax`, `Ay`, `Az`)**: Expresados en aceleración de la gravedad ($g$, donde $1g \approx 9.81\text{ m/s}^2$).
* **Giroscopios (`Gx`, `Gy`, `Gz`)**: Expresados en velocidad angular en grados por segundo ($\circ/\text{s}$).

### Esquema Final de Salida (Capa Oro)
`Dataset`, `Subject`, `Activity_Label`, `Activity_Code`, `Trial`, `Sample_Index`, `Ax`, `Ay`, `Az`, `Gx`, `Gy`, `Gz`, `AVM`, `GVM`


## 1 · Constantes globales


In [1]:
import json
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import pathlib
from scipy.signal import resample_poly, butter, filtfilt, sosfiltfilt
from scipy.stats import pearsonr

# --- Constantes sobre la estructura de los datasets ----------------------------
BRONZE_DIR = pathlib.Path("../data/bronce/falls")
PLATA_DIR = pathlib.Path("../data/plata/falls")
ORO_DIR = pathlib.Path("../data/oro/falls")

# Lista de datasets a procesar
DATASETS_META = {
    "UPFall": {"fs": 100, "csv": "UPFall-Reduced.csv"},
    "KFall": {"fs": 100, "csv": "KFall-Reduced.csv"},
    "FallAllD": {"fs": 238, "csv": "FallAllD-Reduced.csv"},
    "SisFall": {"fs": 200, "csv": "SisFall-Reduced.csv"},
    "UMAFall": {"fs": 20, "csv": "UMAFall-Reduced.csv"},
}

SENSOR_COLS = ["Ax", "Ay", "Az", "Gx", "Gy", "Gz"]
META_COLS = ["Subject", "Activity_Label", "Activity_Code", "Trial"]

# Esquema final de salida (capa oro)
SCHEMA_COLS = [
    "Dataset",
    "Subject",
    "Activity_Label",
    "Activity_Code",
    "Trial",
    "Sample_Index",
    "Ax",
    "Ay",
    "Az",
    "Gx",
    "Gy",
    "Gz",
    "AVM",
    "GVM",
]

# Full-scale ranges (g, °/s) por dataset
ACC_FS = {"SisFall": 16, "FallAllD": 16, "UMAFall": 8, "UPFall": 16, "KFall": 16}
GYRO_FS = {"SisFall": 2000, "FallAllD": 2000, "UMAFall": 256, "UPFall": 2000, "KFall": 2000}

# --- Constantes configurables --------------------------------------------------
FS_TARGET = 50 # Frecuencia de muestreo objetivo (Hz) para el resampleo de todas las señales

# --- Filtrado Butterworth (paso bajo, anti-ruido MEMS) --------------------
BUTTER_ORDER = 4    # Orden del filtro Butterworth
BUTTER_CUTOFF = 8.0 # Frecuencia de corte (Hz) — preserva movimiento humano (<20 Hz)

print("Configuración del Pipeline:")
print(f"  Frecuencia objetivo (FS_TARGET)      : {FS_TARGET} Hz")
print(f"  Datasets a procesar                  : {list(DATASETS_META.keys())}")
print(f"  Esquema final ({len(SCHEMA_COLS)} columnas): {SCHEMA_COLS}")

Configuración del Pipeline:
  Frecuencia objetivo (FS_TARGET)      : 50 Hz
  Datasets a procesar                  : ['UPFall', 'KFall', 'FallAllD', 'SisFall', 'UMAFall']
  Esquema final (14 columnas): ['Dataset', 'Subject', 'Activity_Label', 'Activity_Code', 'Trial', 'Sample_Index', 'Ax', 'Ay', 'Az', 'Gx', 'Gy', 'Gz', 'AVM', 'GVM']


## 2 · Funciones del pipeline

Definición completa de toda la lógica; carga, resampleo, métricas de fidelidad, validación, filtrado y persistencia.


In [2]:
# --- Funciones de carga y guardado de datos --------------------------------------
def load_csv_local(name: str) -> pd.DataFrame:
    """Lee un CSV desde la capa bronce."""
    return pd.read_csv(BRONZE_DIR / name)

def save_parquet_local(df: pd.DataFrame, name: str) -> int:
    """Guarda un DataFrame como Parquet en la capa oro (index=False)."""
    ORO_DIR.mkdir(parents=True, exist_ok=True)
    path = ORO_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    return path.stat().st_size

def save_csv_local(df: pd.DataFrame, name: str) -> int:
    """Guarda un DataFrame como CSV en la capa plata (index=False)."""
    PLATA_DIR.mkdir(parents=True, exist_ok=True)
    path = PLATA_DIR / name
    df.to_csv(path, index=False)
    return path.stat().st_size

def save_json_local(data: dict, name: str) -> int:
    """Guarda un diccionario como JSON en la capa plata."""
    PLATA_DIR.mkdir(parents=True, exist_ok=True)
    path = PLATA_DIR / name
    path.write_text(json.dumps(data, indent=2, default=str), encoding="utf-8")
    return path.stat().st_size

def clear_oro() -> None:
    """Elimina los Parquet existentes en la capa oro (regeneración limpia)."""
    if not ORO_DIR.exists():
        return
    for p in ORO_DIR.glob("*.parquet"):
        p.unlink()

print("✓ Funciones de carga y guardado de datos cargadas.")

✓ Funciones de carga y guardado de datos cargadas.


In [3]:
# --- Auditoría de calidad de datos ------------------------------------------------
def _sat(df: pd.DataFrame, cols: list[str], fs: float) -> dict:
    """Fracción de muestras por canal en el 99% superior del rango full-scale."""
    return {f"{c}_sat_frac": float((df[c].abs() >= 0.99 * fs).mean()) for c in cols}

def saturation_report(df: pd.DataFrame, ds_name: str) -> dict:
    """Fracciones de saturación/clipping por diferencia de full-scale."""
    return {
        **_sat(df, ["Ax", "Ay", "Az"], ACC_FS[ds_name]),
        **_sat(df, ["Gx", "Gy", "Gz"], GYRO_FS[ds_name]),
    }

def check_sanity(df: pd.DataFrame, ds_name: str) -> dict:
    """Auditoría de unidades físicas: NaNs, gravedad, canales muertos y saturación."""
    nan = int(df[SENSOR_COLS].isna().sum().sum())
    avm = np.sqrt(df["Ax"] ** 2 + df["Ay"] ** 2 + df["Az"] ** 2)
    avm_median_g = float(avm.median())
    dead_channels = [c for c in SENSOR_COLS if df[c].std() == 0]
    return {
        "nan": nan,
        "avm_median_g": avm_median_g,
        "dead_channels": dead_channels,
        **saturation_report(df, ds_name),
    }

print("✓ Funciones de auditoría de calidad de datos cargadas.")

✓ Funciones de auditoría de calidad de datos cargadas.


In [4]:
# --- Filtrado Butterworth paso bajo -----------------------------------------
def butter_lowpass_filter(
    signal: np.ndarray,
    fs: int,
    cutoff: float = BUTTER_CUTOFF,
    order: int = BUTTER_ORDER,
) -> np.ndarray:
    """Aplica filtro Butterworth paso bajo con fase cero (sosfiltfilt).

    Se aplica ANTES del resampleo para eliminar ruido de alta frecuencia
    de sensores MEMS, preservando la banda útil del movimiento humano.
    """
    nyq = fs / 2.0
    if cutoff >= nyq * 0.99:
        return signal.copy()
    sos = butter(order, cutoff / nyq, btype="low", output="sos")
    return sosfiltfilt(sos, signal)

# --- Resampleo de señales ------------------------------------------------
def get_poly_factors(fs_orig: int, fs_target: int = FS_TARGET) -> tuple[int, int]:
    """Devuelve (up, down) reducidos al mínimo común divisor."""
    from math import gcd

    g = gcd(fs_target, fs_orig)
    return fs_target // g, fs_orig // g

KAISER_BETA = 5.0 # Parámetro de la ventana Kaiser para el filtrado de paso bajo
def resample_signal(
    signal: np.ndarray,
    fs_orig: int,
    fs_target: int = FS_TARGET,
    kaiser_beta: float = KAISER_BETA,
) -> np.ndarray:
    """Resamplea un vector 1D usando resample_poly y ventana Kaiser."""
    up, down = get_poly_factors(fs_orig, fs_target)
    # padtype="line" reduce el efecto de borde en los extremos del resampleo.
    return resample_poly(signal, up=up, down=down, window=("kaiser", kaiser_beta), padtype="line")

def resample_trial_df(
    trial_df: pd.DataFrame,
    fs_orig: int,
    ds_name: str,
    fs_target: int = FS_TARGET,
    kaiser_beta: float = KAISER_BETA,
    schema_cols: list[str] | None = None,
) -> pd.DataFrame:
    """Resamplea los 6 canales crudos y deriva AVM/GVM, con esquema de oro."""
    if schema_cols is None:
        schema_cols = SCHEMA_COLS

    meta = {c: trial_df[c].iloc[0] for c in META_COLS if c in trial_df.columns}

    # Paso 1: Filtrado Butterworth paso bajo (anti-ruido MEMS)
    filtered = {
        c: butter_lowpass_filter(trial_df[c].astype(float).values, fs_orig)
        for c in SENSOR_COLS
    }
    # Paso 2: Resampleo polyphase con Kaiser
    rs = {
        c: resample_signal(filtered[c], fs_orig, fs_target, kaiser_beta)
        for c in SENSOR_COLS
    }

    avm = np.sqrt(rs["Ax"] ** 2 + rs["Ay"] ** 2 + rs["Az"] ** 2)
    gvm = np.sqrt(rs["Gx"] ** 2 + rs["Gy"] ** 2 + rs["Gz"] ** 2)

    out = pd.DataFrame({**rs, "AVM": avm, "GVM": gvm})
    out["Dataset"] = ds_name
    for c, val in meta.items():
        out[c] = val
    out["Sample_Index"] = np.arange(len(out))

    return out[schema_cols]

print("✓ Funciones de resampleo de señales cargadas.")

✓ Funciones de resampleo de señales cargadas.


In [5]:
# --- Filtrado de trials válidos ------------------------------------------------
# Umbrales de descarte de calidad (lógica OR sobre AVM)
THR_PEARSON_MIN   = 0.85  # Umbral mínimo de correlación de Pearson entre señal original y resampleada
THR_PHASE_MS_MAX  = 100.0 # Umbral máximo de desfase de pico entre señal original y resampleada (ms)
THR_ATTEN_PCT_MAX = 25.0  # Umbral máximo de atenuación de pico entre señal original y resampleada (%)

def filter_valid_trials(
    df_raw: pd.DataFrame,
    df_metrics: pd.DataFrame,
    pearson_min: float = THR_PEARSON_MIN,
    phase_ms_max: float = THR_PHASE_MS_MAX,
    atten_pct_max: float = THR_ATTEN_PCT_MAX,
) -> tuple[list, int, int]:
    """
    Cruza trials de clase Fall con sus métricas de fidelidad (AVM y GVM) y
    filtra los que no cumplen los umbrales de calidad.

    Política OR>=2: una métrica falla si falla en AVM o en GVM; un trial se
    descarta solo si al menos 2 de las 3 métricas fallan. Los trials cortos
    dejan métricas NaN (no evaluables) y se descartan directamente.
    """
    falls = df_raw[df_raw["Activity_Label"] == "Fall"]
    trial_ids = (
        falls[["Subject", "Activity_Code", "Trial"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    pivoted = df_metrics.pivot_table(
        index=["Subject", "Activity_Code", "Trial"],
        columns="sensor",
        values=["pearson_r", "phase_shift_ms", "peak_atten_pct"],
        aggfunc="first",
    )
    pivoted.columns = [f"{m}_{s}" for m, s in pivoted.columns]

    merged = trial_ids.merge(
        pivoted.reset_index(), on=["Subject", "Activity_Code", "Trial"], how="left"
    )

    # Fallo por métrica: falla si no cumple el umbral en AVM O en GVM (NaN = sin evidencia).
    p_fail = (
        (merged["pearson_r_AVM"] < pearson_min) | (merged["pearson_r_GVM"] < pearson_min)
    ).fillna(False).astype(int)
    ph_fail = (
        (merged["phase_shift_ms_AVM"] > phase_ms_max)
        | (merged["phase_shift_ms_GVM"] > phase_ms_max)
    ).fillna(False).astype(int)
    at_fail = (
        (merged["peak_atten_pct_AVM"] > atten_pct_max)
        | (merged["peak_atten_pct_GVM"] > atten_pct_max)
    ).fillna(False).astype(int)
    # Cuenta de fallos: >=2 de 3 métricas malas -> descarte.
    weak = p_fail + ph_fail + at_fail
    # NaN en pearson_r_AVM = trial corto/no evaluable -> descarte directo.
    discard_mask = merged["pearson_r_AVM"].isna() | (weak >= 2)

    valid_ids = merged.loc[
        ~discard_mask, ["Subject", "Activity_Code", "Trial"]
    ].values.tolist()
    n_valid = len(valid_ids)
    n_discarded = int(discard_mask.sum())

    return valid_ids, n_valid, n_discarded

print("✓ Funciones de filtrado de trials válidos cargadas.")

✓ Funciones de filtrado de trials válidos cargadas.


In [6]:
# --- Métricas de fidelidad de resampleo ----------------------------------------
def lowpass(
    signal: np.ndarray,
    fs: int,
    cutoff: float | None = None,
    fs_target: int = FS_TARGET,
) -> np.ndarray:
    """
    Aplica filtro pasa-bajos Butterworth a la señal original para igualar el ancho
    de banda del objetivo antes de calcular métricas de fidelidad.
    """
    if cutoff is None:
        cutoff = fs_target / 2.0 - 0.5
    nyq = fs / 2.0
    if cutoff >= nyq * 0.99:
        return signal.copy()
    b, a = butter(8, cutoff / nyq, btype="low")
    return filtfilt(b, a, signal)

def snr_inband(
    orig: np.ndarray,
    resampled: np.ndarray,
    fs_orig: int,
    fs_target: int = FS_TARGET,
) -> float:
    """Calcula SNR (dB) en la banda [0, FS_TARGET/2] Hz."""
    t_orig = np.linspace(0, 1, len(orig))
    t_res  = np.linspace(0, 1, len(resampled))
    resampled_interp = np.interp(t_orig, t_res, resampled)
    orig_filtered = lowpass(orig, fs_orig, fs_target=fs_target)
    noise = orig_filtered - resampled_interp
    power_signal = np.mean(orig_filtered ** 2)
    power_noise  = np.mean(noise ** 2)
    if power_noise == 0:
        return float("inf")
    return float(10 * np.log10(power_signal / power_noise))

def pearson_inband(
    orig: np.ndarray,
    resampled: np.ndarray,
    fs_orig: int,
    fs_target: int = FS_TARGET,
) -> float:
    """Calcula correlación de Pearson entre original filtrada y resampleada."""
    t_orig = np.linspace(0, 1, len(orig))
    t_res  = np.linspace(0, 1, len(resampled))
    resampled_interp = np.interp(t_orig, t_res, resampled)
    orig_filtered = lowpass(orig, fs_orig, fs_target=fs_target)
    r, _ = pearsonr(orig_filtered, resampled_interp)
    return float(r)

def peak_phase_shift_ms(
    orig: np.ndarray,
    resampled: np.ndarray,
    fs_orig: int,
    fs_target: int = FS_TARGET,
) -> float:
    """Calcula desfase temporal del pico en milisegundos."""
    return float(abs(orig.argmax() / fs_orig - resampled.argmax() / fs_target) * 1000.0)

def peak_attenuation_pct(
    orig: np.ndarray,
    resampled: np.ndarray,
    fs_orig: int,
    fs_target: int = FS_TARGET,
) -> float:
    """Calcula atenuación porcentual del pico relativo a la original filtrada."""
    orig_filtered = lowpass(orig, fs_orig, fs_target=fs_target)
    max_orig = np.max(orig_filtered)
    t_orig = np.linspace(0, 1, len(orig))
    t_res  = np.linspace(0, 1, len(resampled))
    max_res = np.max(np.interp(t_orig, t_res, resampled))
    if max_orig > 0:
        return float(abs(max_orig - max_res) / max_orig * 100.0)
    return 0.0

EVAL_WINDOW_SEC = 2.0 # Ventana de evaluación de métricas (en segundos)
def descriptive_stats(signal: np.ndarray) -> dict:
    """Estadísticos descriptivos de señal completa (comparación pre/post resampleo)."""
    return {
        "mean":   float(np.mean(signal)),
        "std":    float(np.std(signal)),
        "median": float(np.median(signal)),
        "p05":    float(np.percentile(signal, 5)),
        "p95":    float(np.percentile(signal, 95)),
    }

def analyze_trial(
    trial_df: pd.DataFrame,
    fs_orig: int,
    fs_target: int = FS_TARGET,
    kaiser_beta: float = KAISER_BETA,
    eval_window_sec: float = EVAL_WINDOW_SEC,
) -> dict | None:
    """Calcula métricas de fidelidad de resampleo y estadísticos pre/post para AVM y GVM."""
    avm = np.sqrt(
        trial_df["Ax"] ** 2 + trial_df["Ay"] ** 2 + trial_df["Az"] ** 2
    ).values
    gvm = np.sqrt(
        trial_df["Gx"] ** 2 + trial_df["Gy"] ** 2 + trial_df["Gz"] ** 2
    ).values

    if len(avm) < fs_orig:
        return None

    avm_r = resample_signal(avm, fs_orig, fs_target, kaiser_beta)
    gvm_r = resample_signal(gvm, fs_orig, fs_target, kaiser_beta)

    w      = int(eval_window_sec / 2.0 * fs_target)
    peak_r = avm_r.argmax()
    s, e   = max(0, peak_r - w), min(len(avm_r), peak_r + w)

    wo     = int(eval_window_sec / 2.0 * fs_orig)
    peak_o = avm.argmax()
    so, eo = max(0, peak_o - wo), min(len(avm), peak_o + wo)

    result = {}
    for sensor, orig, res, ow, rw in [
        ("AVM", avm, avm_r, avm[so:eo], avm_r[s:e]),
        ("GVM", gvm, gvm_r, gvm[so:eo], gvm_r[s:e]),
    ]:
        result[sensor] = {
            "snr_db":         snr_inband(ow, rw, fs_orig, fs_target),
            "pearson_r":      pearson_inband(ow, rw, fs_orig, fs_target),
            "phase_shift_ms": peak_phase_shift_ms(ow, rw, fs_orig, fs_target),
            "peak_atten_pct": peak_attenuation_pct(ow, rw, fs_orig, fs_target),
            **{f"orig_{k}": v for k, v in descriptive_stats(orig).items()},
            **{f"res_{k}":  v for k, v in descriptive_stats(res).items()},
        }
    return result

def run_trial_metrics(
    df: pd.DataFrame,
    fs_orig: int,
    fs_target: int = FS_TARGET,
    label: str = "Fall",
) -> pd.DataFrame:
    """Itera por groupby sobre los trials de la clase indicada y calcula métricas.
    """
    falls = df[df["Activity_Label"] == label]
    rows  = []
    groups = list(
        falls.groupby(["Subject", "Activity_Code", "Trial"], sort=False)
    )
    n_done = 0
    for (subj, code, trial), grp in groups:
        res = analyze_trial(grp, fs_orig, fs_target)
        n_done += 1
        if res is None:
            continue
        for sensor in ("AVM", "GVM"):
            rows.append({
                "Subject":       subj,
                "Activity_Code": code,
                "Trial":         trial,
                "sensor":        sensor,
                **res[sensor],
            })
        if n_done % 200 == 0 or n_done == len(groups):
            print(f"  Progreso: {n_done}/{len(groups)} trials", end="\r")
    print()
    return pd.DataFrame(rows)

print("✓ Funciones de métricas de fidelidad de resampleo cargadas.")


✓ Funciones de métricas de fidelidad de resampleo cargadas.


In [7]:
# --- Validación de etiquetas y esquema ----------------------
VALID_LABELS = {"Fall", "ADL"}

def validate_labels(
    df: pd.DataFrame, valid_labels: set[str] | None = None
) -> dict:
    """Verifica que Activity_Label contenga solo valores esperados y sin mezcla en trials."""
    if valid_labels is None:
        valid_labels = VALID_LABELS
    unexpected    = set(df["Activity_Label"].unique()) - valid_labels
    trial_counts  = (
        df.groupby(["Subject", "Activity_Code", "Trial"])["Activity_Label"].nunique()
    )
    return {
        "unexpected_labels": unexpected,
        "mixed_trials":      trial_counts[trial_counts > 1],
    }

def validate_schema(df: pd.DataFrame, schema_cols: list[str] | None = None) -> bool:
    """Verifica que el DataFrame tenga exactamente las columnas en el orden indicado."""
    if schema_cols is None:
        schema_cols = SCHEMA_COLS
    return list(df.columns) == schema_cols

print("✓ Funciones de validación y filtrado cargadas.")

✓ Funciones de validación y filtrado cargadas.


## 3 · Ingesta desde Capa Bronce


In [8]:
raw_datasets: dict[str, pd.DataFrame] = {}

print("Cargando datasets crudos desde bronce...")
for ds_name, meta in DATASETS_META.items():
    try:
        df = load_csv_local(meta["csv"])
        raw_datasets[ds_name] = df
        print(f"  ✓ {ds_name:10s} ({meta['fs']} Hz) → {len(df):>10,} filas cargadas.")
    except Exception as err:
        print(f"  ✗ Error al cargar {ds_name}: {err}")

print(f"\nDatasets listos en memoria: {list(raw_datasets.keys())}")


Cargando datasets crudos desde bronce...


  ✓ UPFall     (100 Hz) →    294,678 filas cargadas.


  ✓ KFall      (100 Hz) →  3,995,100 filas cargadas.


  ✓ FallAllD   (238 Hz) →  8,558,480 filas cargadas.


  ✓ SisFall    (200 Hz) → 15,858,929 filas cargadas.


  ✓ UMAFall    (20 Hz) →    164,392 filas cargadas.

Datasets listos en memoria: ['UPFall', 'KFall', 'FallAllD', 'SisFall', 'UMAFall']


In [9]:

print("Auditoria de unidades fisicas y saturacion (por diferencia de full-scale):")
sanity_reports = {n: check_sanity(df, n) for n, df in raw_datasets.items()}

for n, r in sanity_reports.items():
    sat = {k: round(v, 4) for k, v in r.items() if k.endswith("_sat_frac")}
    print(f"  {n:10s} | avm_median_g={r["avm_median_g"]:.4f} | nan={r["nan"]} | dead={r["dead_channels"]} | sat={sat}")

save_json_local(sanity_reports, "bronze_sanity.json")

for n, r in sanity_reports.items():
    if not (0.75 <= r["avm_median_g"] <= 1.35) or r["nan"] > 0:
        raise ValueError(f"Unidades/sanity invalidos en {n}")

print("\nAuditoria de unidades completada sin anomalias.")


Auditoria de unidades fisicas y saturacion (por diferencia de full-scale):


  UPFall     | avm_median_g=1.0114 | nan=0 | dead=[] | sat={'Ax_sat_frac': 0.0, 'Ay_sat_frac': 0.0, 'Az_sat_frac': 0.0, 'Gx_sat_frac': 0.0, 'Gy_sat_frac': 0.0001, 'Gz_sat_frac': 0.0}
  KFall      | avm_median_g=1.0081 | nan=0 | dead=[] | sat={'Ax_sat_frac': 0.0, 'Ay_sat_frac': 0.0, 'Az_sat_frac': 0.0, 'Gx_sat_frac': 0.0, 'Gy_sat_frac': 0.0, 'Gz_sat_frac': 0.0}
  FallAllD   | avm_median_g=0.9958 | nan=0 | dead=[] | sat={'Ax_sat_frac': 0.0, 'Ay_sat_frac': 0.0, 'Az_sat_frac': 0.0, 'Gx_sat_frac': 0.0, 'Gy_sat_frac': 0.0, 'Gz_sat_frac': 0.0}
  SisFall    | avm_median_g=0.9991 | nan=0 | dead=[] | sat={'Ax_sat_frac': 0.0, 'Ay_sat_frac': 0.0, 'Az_sat_frac': 0.0, 'Gx_sat_frac': 0.0, 'Gy_sat_frac': 0.0, 'Gz_sat_frac': 0.0}
  UMAFall    | avm_median_g=1.0099 | nan=0 | dead=[] | sat={'Ax_sat_frac': 0.0002, 'Ay_sat_frac': 0.0, 'Az_sat_frac': 0.0001, 'Gx_sat_frac': 0.0006, 'Gy_sat_frac': 0.0073, 'Gz_sat_frac': 0.0007}

Auditoria de unidades completada sin anomalias.


## 4 · Validación de Integridad de Etiquetas

Verifica que la columna `Activity_Label` contenga únicamente los valores válidos (`Fall` y `ADL`) y detecta trials anómalos con mezcla incoherente de etiquetas.


In [10]:
label_summary_rows = []

print("Validando integridad de etiquetas...")
for ds_name, df in raw_datasets.items():
    res = validate_labels(df)

    if res["unexpected_labels"]:
        print(f"  ⚠️ [{ds_name}] Etiquetas inesperadas: {res['unexpected_labels']}")
    else:
        print(f"  ✅ [{ds_name}] Etiquetas válidas exclusivamente (Fall/ADL).")

    if len(res["mixed_trials"]) > 0:
        print(f"  ⚠️ [{ds_name}] {len(res['mixed_trials'])} trial(s) con mezcla incoherente.")
    else:
        print(f"  ✅ [{ds_name}] Sin mezcla de etiquetas en trials.")

    trial_labels = df.groupby(["Subject", "Activity_Code", "Trial"])["Activity_Label"].first()
    n_fall = (trial_labels == "Fall").sum()
    n_adl  = (trial_labels == "ADL").sum()
    label_summary_rows.append({
        "Dataset": ds_name, "ADL": n_adl, "Fall": n_fall,
        "Total Trials": n_fall + n_adl
    })

print("\nDistribución inicial de trials por dataset:")
print(pd.DataFrame(label_summary_rows).set_index("Dataset").to_string())


Validando integridad de etiquetas...
  ✅ [UPFall] Etiquetas válidas exclusivamente (Fall/ADL).
  ✅ [UPFall] Sin mezcla de etiquetas en trials.


  ✅ [KFall] Etiquetas válidas exclusivamente (Fall/ADL).
  ✅ [KFall] Sin mezcla de etiquetas en trials.


  ✅ [FallAllD] Etiquetas válidas exclusivamente (Fall/ADL).
  ✅ [FallAllD] Sin mezcla de etiquetas en trials.


  ✅ [SisFall] Etiquetas válidas exclusivamente (Fall/ADL).
  ✅ [SisFall] Sin mezcla de etiquetas en trials.


  ✅ [UMAFall] Etiquetas válidas exclusivamente (Fall/ADL).
  ✅ [UMAFall] Sin mezcla de etiquetas en trials.

Distribución inicial de trials por dataset:
           ADL  Fall  Total Trials
Dataset                           
UPFall     304   255           559
KFall     2729  2346          5075
FallAllD  1332   466          1798
SisFall   2702  1798          4500
UMAFall    373   180           553


## 5 · Generación de Métricas de Fidelidad por Trial (Capa Plata)

Se calculan las cinco métricas de fidelidad de resampleo sobre AVM y GVM. Los resultados se persisten en `plata/falls`.


In [11]:
metrics_results: dict[str, pd.DataFrame] = {}
metrics_frames = []

for ds_name, meta in DATASETS_META.items():
    if ds_name not in raw_datasets:
        continue

    fs_orig = meta["fs"]
    if fs_orig == FS_TARGET:
        print(f"[{ds_name}] {fs_orig} Hz = objetivo. Sin resampleo real, se omiten métricas.")
        continue

    print(f"\n[{ds_name}] Calculando métricas de fidelidad ({fs_orig} Hz → {FS_TARGET} Hz)...")
    df_mets = run_trial_metrics(raw_datasets[ds_name], fs_orig=fs_orig, fs_target=FS_TARGET)
    metrics_results[ds_name] = df_mets
    metrics_frames.append(df_mets.assign(Dataset=ds_name))
    print(f"  ✓ {len(df_mets):,} filas de métricas generadas.")

df_all_metrics = pd.concat(metrics_frames, ignore_index=True) if metrics_frames else pd.DataFrame()

if not df_all_metrics.empty:
    csv_filename  = f"resampling_metrics_per_trial_{FS_TARGET}hz.csv"
    bytes_written = save_csv_local(df_all_metrics, csv_filename)
    print(f"\n✅ {csv_filename} → data/plata/falls/ ({bytes_written:,} bytes, {len(df_all_metrics):,} filas).")



[UPFall] Calculando métricas de fidelidad (100 Hz → 50 Hz)...


  Progreso: 255/255 trials
  ✓ 510 filas de métricas generadas.

[KFall] Calculando métricas de fidelidad (100 Hz → 50 Hz)...


  Progreso: 2346/2346 trials
  ✓ 4,692 filas de métricas generadas.

[FallAllD] Calculando métricas de fidelidad (238 Hz → 50 Hz)...


  Progreso: 466/466 trials
  ✓ 932 filas de métricas generadas.

[SisFall] Calculando métricas de fidelidad (200 Hz → 50 Hz)...


  Progreso: 1798/1798 trials
  ✓ 3,596 filas de métricas generadas.

[UMAFall] Calculando métricas de fidelidad (20 Hz → 50 Hz)...


  Progreso: 180/180 trials
  ✓ 360 filas de métricas generadas.



✅ resampling_metrics_per_trial_50hz.csv → data/plata/falls/ (2,783,831 bytes, 10,090 filas).


## 6 · Filtrado de Calidad por Trial y Persistencia de Configuración

Se aplican los umbrales de descarte sobre las magnitudes vectoriales AVM y GVM (una métrica falla si no cumple el umbral en cualquiera de los dos):
- Pearson $r \ge 0.85$
- Desfase de pico $\le 100\text{ ms}$
- Atenuación de pico $\le 25\%$

Se sube `trial_quality_config.json` a `plata/falls/` con la lista de IDs de trials válidos por dataset.


In [12]:
quality_config = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "fs_target": FS_TARGET,
    "criteria": {
        "pearson_r_min":      THR_PEARSON_MIN,
        "phase_shift_ms_max": THR_PHASE_MS_MAX,
        "peak_atten_pct_max": THR_ATTEN_PCT_MAX,
        "sensors":            ["AVM", "GVM"],
        "combine":            "or",
    },
    "datasets": {}
}

print("Filtrando trials según criterios de calidad de resampleo:\n")
print(f"  {'Dataset':12s}  {'Total Fall':>10}  {'Válidos':>9}  {'Descartados':>12}  {'% Descarte':>10}")
print("  " + "-" * 58)

for ds_name, meta in DATASETS_META.items():
    if ds_name not in raw_datasets:
        continue

    df_raw = raw_datasets[ds_name]
    falls  = df_raw[df_raw["Activity_Label"] == "Fall"]
    total_falls = falls[["Subject", "Activity_Code", "Trial"]].drop_duplicates().shape[0]

    if ds_name not in metrics_results:
        valid_ids   = falls[["Subject", "Activity_Code", "Trial"]].drop_duplicates().values.tolist()
        n_valid, n_discarded = total_falls, 0
    else:
        df_mets = metrics_results[ds_name].reset_index(drop=True)
        valid_ids, n_valid, n_discarded = filter_valid_trials(
            df_raw, df_mets,
            pearson_min=THR_PEARSON_MIN,
            phase_ms_max=THR_PHASE_MS_MAX,
            atten_pct_max=THR_ATTEN_PCT_MAX
        )

    pct_discard = (n_discarded / total_falls * 100) if total_falls > 0 else 0.0
    quality_config["datasets"][ds_name] = {
        "total": total_falls, "valid": n_valid,
        "discarded": n_discarded, "valid_ids": valid_ids,
    }
    print(f"  {ds_name:12s}  {total_falls:>10,}  {n_valid:>9,}  {n_discarded:>12,}  {pct_discard:>9.1f}%")

json_bytes = save_json_local(quality_config, "trial_quality_config.json")
print(f"\n✅ trial_quality_config.json → data/plata/falls/ ({json_bytes:,} bytes).")

Filtrando trials según criterios de calidad de resampleo:

  Dataset       Total Fall    Válidos   Descartados  % Descarte
  ----------------------------------------------------------
  UPFall               255        252             3        1.2%


  KFall              2,346      2,295            51        2.2%


  FallAllD             466        415            51       10.9%


  SisFall            1,798      1,742            56        3.1%
  UMAFall              180        163            17        9.4%

✅ trial_quality_config.json → data/plata/falls/ (355,155 bytes).


## 7 · Resampleo Definitivo a 50 Hz y Exportación a Capa Oro

Por cada dataset se calcula AVM/GVM, se resamplea con filtro Kaiser y se exporta con el esquema de 14 columnas:
`Dataset`, `Subject`, `Activity_Label`, `Activity_Code`, `Trial`, `Sample_Index`, `Ax`, `Ay`, `Az`, `Gx`, `Gy`, `Gz`, `AVM`, `GVM`

Se exportan dos archivos consolidados: 
* **SET *A***: `oro/falls/set_a.parquet` (5 datasets)
* **SET *B***: `oro/falls/set_b.parquet` (4 datasets, sin UMAFall).

In [13]:
print("Procesando resampleo definitivo y exportacion a Parquet en oro/falls/ (50 Hz):\n")
resampled_by_ds: dict[str, pd.DataFrame] = {}
for ds_name in list(DATASETS_META.keys()):
    if ds_name not in raw_datasets or ds_name not in quality_config["datasets"]:
        continue
    fs_orig  = DATASETS_META[ds_name]["fs"]
    df_raw   = raw_datasets.pop(ds_name)
    
    valid_set = {
        (row[0], row[1], row[2])
        for row in quality_config["datasets"][ds_name]["valid_ids"]
    }
    df_fall = df_raw[df_raw["Activity_Label"] == "Fall"]
    df_adl  = df_raw[df_raw["Activity_Label"] == "ADL"]
    fall_keys = pd.MultiIndex.from_frame(
        df_fall[["Subject", "Activity_Code", "Trial"]]
    )
    valid_mi = pd.MultiIndex.from_tuples(valid_set)
    df_fall_valid = df_fall[fall_keys.isin(valid_mi)]
    df_to_resample = pd.concat([df_fall_valid, df_adl], ignore_index=True)
    del df_raw, df_fall, df_adl, df_fall_valid
    
    resampled_chunks = []
    for _, grp in df_to_resample.groupby(["Subject", "Activity_Code", "Trial"], sort=False):
        grp_copy = grp.copy()
        for col in SENSOR_COLS:
            if col in grp_copy.columns:
                grp_copy[col] = grp_copy[col].astype("float32")
        resampled_chunks.append(
            resample_trial_df(
                grp_copy, fs_orig=fs_orig, fs_target=FS_TARGET,
                kaiser_beta=KAISER_BETA, schema_cols=SCHEMA_COLS, ds_name=ds_name,
            )
        )
    del df_to_resample
    
    df_out = pd.concat(resampled_chunks, ignore_index=True)
    del resampled_chunks
    
    if not validate_schema(df_out, SCHEMA_COLS):
        raise ValueError(f"[{ds_name}] El esquema final no coincide con el orden esperado de 14 columnas.")
    n_fall_rows = (df_out["Activity_Label"] == "Fall").sum()
    n_adl_rows  = (df_out["Activity_Label"] == "ADL").sum()
    print(f"  \u2713 [{ds_name:10s}] {fs_orig} Hz \u2192 {FS_TARGET} Hz")
    print(f"    Fall: {n_fall_rows:>9,} filas | ADL: {n_adl_rows:>9,} filas | Total: {len(df_out):>9,}")
    resampled_by_ds[ds_name] = df_out
    del df_out
    
# --- Consolidacion en 2 conjuntos de oro a 50 Hz ---------------------------------
clear_oro()
set_a = pd.concat([resampled_by_ds[d] for d in DATASETS_META], ignore_index=True)
set_b = pd.concat(
    [resampled_by_ds[d] for d in DATASETS_META if d != "UMAFall"], ignore_index=True
)
# Columnas que mezclan int/str entre datasets (Subject, Activity_Code) -> str.
for col in set_a.columns:
    if set_a[col].dtype == object:
        set_a[col] = set_a[col].astype(str)
for col in set_b.columns:
    if set_b[col].dtype == object:
        set_b[col] = set_b[col].astype(str)
bytes_a = save_parquet_local(set_a, "set_a")
bytes_b = save_parquet_local(set_b, "set_b")
print(f"\n  \u2713 set_a.parquet (5 datasets @ {FS_TARGET} Hz): {bytes_a / (1024**2):.2f} MB")
print(f"  \u2713 set_b.parquet (4 datasets @ {FS_TARGET} Hz, sin UMAFall): {bytes_b / (1024**2):.2f} MB")
print("=" * 65)
print("  Pipeline ETL finalizado exitosamente. Capa Oro lista para uso.")
print("=" * 65)


Procesando resampleo definitivo y exportacion a Parquet en oro/falls/ (50 Hz):



  ✓ [UPFall    ] 100 Hz → 50 Hz
    Fall:    22,765 filas | ADL:   124,443 filas | Total:   147,208


  ✓ [KFall     ] 100 Hz → 50 Hz
    Fall:   844,349 filas | ADL: 1,135,503 filas | Total: 1,979,852


  ✓ [FallAllD  ] 238 Hz → 50 Hz
    Fall:   415,000 filas | ADL: 1,332,000 filas | Total: 1,747,000


  ✓ [SisFall   ] 200 Hz → 50 Hz
    Fall: 1,306,463 filas | ADL: 2,616,399 filas | Total: 3,922,862


  ✓ [UMAFall   ] 20 Hz → 50 Hz
    Fall:   121,449 filas | ADL:   277,188 filas | Total:   398,637



  ✓ set_a.parquet (5 datasets @ 50 Hz): 521.53 MB
  ✓ set_b.parquet (4 datasets @ 50 Hz, sin UMAFall): 496.77 MB
  Pipeline ETL finalizado exitosamente. Capa Oro lista para uso.


## 8 · Evidencia y Justificación Técnica de la Frecuencia Objetivo (50 Hz)

Resumen estadístico de las métricas de fidelidad de señal calculadas internamente durante la ejecución del pipeline sobre AVM y GVM, más la comparación de estadísticos descriptivos antes y después del resampleo:


In [14]:
if not df_all_metrics.empty:
    summary_list = []
    for (ds_name, sensor), grp in df_all_metrics.groupby(["Dataset", "sensor"]):
        summary_list.append({
            "Dataset":                     ds_name,
            "Sensor":                      sensor,
            "Pearson r (Mediana)":          grp["pearson_r"].median(),
            "Pearson r (P05)":              grp["pearson_r"].quantile(0.05),
            "Desfase pico (Mediana ms)":    grp["phase_shift_ms"].median(),
            "Desfase pico (P95 ms)":        grp["phase_shift_ms"].quantile(0.95),
            "Atenuación pico (Mediana %)": grp["peak_atten_pct"].median(),
            "Atenuación pico (P95 %)":     grp["peak_atten_pct"].quantile(0.95),
        })
    print("Resumen numérico de fidelidad de señal a 50 Hz (AVM y GVM):")
    print(pd.DataFrame(summary_list).set_index(["Dataset", "Sensor"]).round(3).to_string())

    print("\nComparación de estadísticos descriptivos pre/post resampleo (mediana por trial):")
    desc_rows = []
    for (ds_name, sensor), grp in df_all_metrics.groupby(["Dataset", "sensor"]):
        row = {"Dataset": ds_name, "Sensor": sensor}
        for stat in ["mean", "std", "median", "p05", "p95"]:
            o    = grp[f"orig_{stat}"].median()
            r    = grp[f"res_{stat}"].median()
            diff = abs(o - r) / o * 100.0 if o else 0.0
            row[f"{stat} orig"] = o
            row[f"{stat} res"]  = r
            row[f"{stat} Δ%"]   = diff
        desc_rows.append(row)
    print(pd.DataFrame(desc_rows).set_index(["Dataset", "Sensor"]).round(3).to_string())
else:
    print("No se requirió resampleo (todos los datasets tienen frecuencia nativa a 50 Hz).")


Resumen numérico de fidelidad de señal a 50 Hz (AVM y GVM):
                 Pearson r (Mediana)  Pearson r (P05)  Desfase pico (Mediana ms)  Desfase pico (P95 ms)  Atenuación pico (Mediana %)  Atenuación pico (P95 %)
Dataset  Sensor                                                                                                                                              
FallAllD AVM                   0.903            0.278                        0.0                  0.000                        4.653                   16.102
         GVM                   0.959            0.275                       10.0                397.857                        3.399                   16.310
KFall    AVM                   0.986            0.788                        0.0                  0.000                        4.216                    9.484
         GVM                   0.991            0.824                       10.0                 80.000                        3.890                  

### Conclusión
Las métricas de fidelidad para **AVM y GVM** confirman que **50 Hz** es una frecuencia de muestreo adecuada para el pipeline:

1. **Forma de onda preservada**: la mediana de Pearson $r$ es $\ge 0.90$ en todos los datasets y ambos sensores (AVM: 0.903–0.986; GVM: 0.959–0.991). El resampleo no deforma la señal en la banda útil.
2. **Amplitud controlada**: la atenuación de pico P95 queda bajo el umbral del 25 % en todos los casos (máx. 21.2 % en AVM de UMAFall; 16.9 % en GVM). La magnitud del impacto se conserva.
3. **Estadísticos invariantes**: la comparación pre/post resampleo muestra variación mínima en toda la distribución — $\Delta$ media < 0.4 %, $\Delta$ std < 9 %, $\Delta$ percentiles < 10 %. El resampleo no introduce sesgo ni distorsión estadística apreciable.
4. **Sincronía del impacto**: el desfase del pico de AVM se mantiene $\le 20$ ms (P95). En GVM la mediana es de una muestra (10–20 ms), pero el P95 alcanza 230–398 ms en FallAllD y UMAFall. Ese exceso se concentra en el 5 % de trials con peor desfase y no se refleja en forma ni amplitud (r mediana $\ge 0.96$, atenuación baja), por lo que se atribuye a picos secundarios ambiguos del giroscopio — limitación de la métrica de desfase por `argmax`, no una pérdida real de fidelidad.

**En conjunto**, 50 Hz es una frecuencia válida: la fidelidad es alta en forma, amplitud y estadísticos para ambos sensores. El único matiz es el desfase de pico del GVM en una minoría de trials de FallAllD y UMAFall, donde conviene priorizar la correlación y la atenuación como criterios de calidad.
